# Task description
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import pandas as pd
import numpy as np
import plotly.express as ple
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
from sklearn.mixture import GaussianMixture
from tqdm.notebook import tqdm
import pickle

# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

## 2. Load up NASA 2020 dataset

In [ ]:
magData = pd.read_csv("../../src/datasets/2020_ydhm_id.csv")
magData.index = pd.DatetimeIndex(magData["ydhm_id"])
magData = magData.drop(columns=["ydhm_id"])
# Consider only one month - January
# magData = magData[magData.index < pd.to_datetime("2020-02-01 00:00:00")]
N = magData.shape[0]
dataset = magData.iloc[:N]
# Free space
del magData
# Show dataset
dataset.diff().head(10)

In [ ]:
gmm = dict(
    series=dataset.diff()["Bx"].values,
    window={"size": 540, "step": 1},
    dates=dataset.index,
    num_comp=4,
)
GMMkwargs = dict(
    n_components=gmm["num_comp"],
    covariance_type="spherical",
    random_state=0,
    warm_start=True,
    init_params="k-means++",
    tol=5e-3,
)
# Initialize model on first window and estimate parameters
model = GaussianMixture(**GMMkwargs)

initial_window = gmm["series"][: gmm["window"]["size"]]
__initial_window = initial_window[~np.isnan(initial_window)]  # Get rid of gaps
mixture = model.fit(__initial_window.reshape(-1, 1))
# Preserving parameters
gmm["weights"] = mixture.weights_.reshape(-1, 1)
gmm["means"] = mixture.means_
gmm["variances"] = mixture.covariances_.reshape(-1, 1)

In [ ]:
# # Plot filled values
# df = pd.DataFrame({"filled": np.isnan(initial_window)})
# gaps_number = np.isnan(initial_window).sum()
# fillings, _ = model.sample(gaps_number)
# initial_window[np.isnan(initial_window)] = fillings.reshape(-1)
# df["series"] = initial_window
# ple.scatter(df.iloc[:2000], color="filled").show()

In [ ]:
__start = gmm["window"]["step"]
__stop = len(gmm["series"]) - gmm["window"]["size"]
# __stop = gmm["window"]["step"] + 100
__step = gmm["window"]["step"]

for i in tqdm(range(__start, __stop, __step)):
    # Take current window and fit GMM on it
    values = gmm["series"][i : gmm["window"]["size"] + i].reshape(-1, 1)
    vals = values[~np.isnan(values)].reshape(-1, 1)
    mixture = model.fit(vals)
    # If one of the variances is lower then threshold then create new model
    j = 0
    while min(mixture.covariances_.reshape(-1, 1)) < GMMkwargs["tol"] and j < 20:
        model = GaussianMixture(**GMMkwargs)
        mixture = model.fit(vals)
        j += 1

    # Initialize container for parameters
    gmm["weights"] = np.append(gmm["weights"], mixture.weights_.reshape(-1, 1), axis=1)
    gmm["means"] = np.append(gmm["means"], mixture.means_, axis=1)
    gmm["variances"] = np.append(
        gmm["variances"], mixture.covariances_.reshape(-1, 1), axis=1
    )
    if any(np.isnan(values)):
        gaps_number = np.isnan(values).sum()
        fillings, _ = model.sample(gaps_number)
        values[np.isnan(values)] = fillings.reshape(-1)

In [ ]:
ws = gmm["window"]["size"]
nc = gmm["num_comp"]
with open(f"data/dBx_{ws}_{nc}.pkl", "wb") as f:
    pickle.dump(gmm, f)

In [ ]:
ple.line(
    {i: gmm["variances"][i] for i in range(gmm["num_comp"])}, title="Variances"
).show()
ple.line({i: gmm["weights"][i] for i in range(gmm["num_comp"])}, title="Weights").show()
ple.line({i: gmm["means"][i] for i in range(gmm["num_comp"])}, title="Means").show()